In [0]:
%pip install psycopg>=3.1.0

households_df = spark.table("workspace.silver.households")
members_df = spark.table("workspace.silver.members")

host = dbutils.secrets.get(
    scope="smart-home-secrets",
    key="host"
)
database = "databricks_postgres"
username = dbutils.secrets.get(
    scope="smart-home-secrets",
    key="lakebase-username"
)
password = dbutils.secrets.get(
    scope="smart-home-secrets",
    key="lakebase-password"
)

import psycopg
from io import StringIO

conn = psycopg.connect(
    host=host,
    port=5432,
    dbname=database,
    user=username,
    password=password,
    sslmode="require",
)

with conn.cursor() as cur:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS silver.households (
            household_id INTEGER,
            name TEXT,
            household_size INTEGER,
            city TEXT,
            household_code TEXT,
            weekly_limit_kwh DOUBLE PRECISION,
            monthly_limit_kwh DOUBLE PRECISION
        )
    """)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS silver.members (
            household_id INTEGER,
            member_id TEXT,
            email TEXT,
            name TEXT,
            role TEXT
        )
    """)
conn.commit()

for df_spark, table_name in [
    (households_df, "silver.households"),
    (members_df, "silver.members"),
]:
    pdf = df_spark.toPandas()
    buffer = StringIO()
    pdf.to_csv(buffer, index=False, header=False)
    buffer.seek(0)
    with conn.cursor() as cur:
        cur.execute(f"TRUNCATE TABLE {table_name}")
        with cur.copy(f"COPY {table_name} FROM STDIN WITH (FORMAT csv)") as copy:
            copy.write(buffer.getvalue())
conn.commit()
conn.close()